# OR-mask reprint segments + copy source images with overlays

This notebook does four things:

1. builds a global **OR no-filament mask** from the extractor,
2. converts that into a **missed-print mask** using the G-code,
3. scores **real G-code extrusion segments** and highlights the faulty ones,
4. finds source images that intersect those faulty segments and copies them into a new folder **with the faulty segments drawn on top of the original image**.

The overlay is done in the original image coordinate system. That is the key step for keeping the scaling accurate.

## Geometry used for the overlay

The overlay works because the merge pipeline already defines a consistent coordinate mapping:

- each frame mask is first converted back into the **original image geometry** using `undo_preprocess_mask(...)`;
- that original-image-sized mask is pasted into the global canvas with its **image center** placed at the frame's `(x, y)` pose;
- therefore, to project a global faulty-segment mask back onto one source image, we do the exact inverse:
  - find the frame center `(cx, cy)` in global pixels,
  - crop a window of size `(image_height, image_width)` from the global faulty-segment mask, centered at `(cx, cy)`.

That crop is already aligned with the original image pixels.

So the overlay does **not** guess scale heuristically. It uses the same geometry as the merge itself.

In [ ]:
import os
import re
import math
import shutil
import zipfile
import importlib.util
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

## Configuration

In [ ]:
# -------- Input data --------
FOLDER = r"/mnt/data/work_or/disc2accurate"
GCODE_FILE = r"/mnt/data/P4_one_layer_annular_disc_60OD_12ID_0p20H(3).gcode"

ZIP_FILE = r"/mnt/data/disc2accurate(4).zip"
EXTRACT_TO = r"/mnt/data/work_or"

# -------- Local helper modules --------
FILAMENT_ARRAY_PY = "filament_array.py"
GCODE_COORD_MASK_PY = "gcode_coordinate_mask.py"
MERGE_OR_PY = "merge_miss_print_or.py"

# -------- Extractor / merge settings --------
RADIUS = 35
THRESHOLD = 0.15
PX_PER_MM = 26.13
FLIP_X = False
FLIP_Y = True
SWAP_XY = False
LAYER_INDEX = 0
GCODE_LINE_WIDTH_MM = 0.45
MAX_FRAME_Z = 0.20
MARGIN_PX = 200

# -------- Segment severity settings --------
SEVERITY_MODE = "mean"   # "mean" or "sum"
SEGMENT_SEVERITY_THRESHOLD = 0.10
SEGMENT_DRAW_THICKNESS = 2

# Additional padding used when building the faulty segment search mask.
# This makes image selection less brittle.
FAULTY_SEGMENT_PADDING_PX = 2

# -------- Source-image extraction settings --------
# A frame is selected if its evaluated region overlaps the faulty segment mask enough.
MIN_FAULTY_OVERLAP_PIXELS = 10
MIN_FAULTY_OVERLAP_FRACTION = 0.001
MAX_IMAGES_TO_COPY = None   # e.g. 50, or None for all

# Overlay appearance
OVERLAY_COLOR_BGR = (0, 0, 255)   # red in OpenCV BGR
OVERLAY_ALPHA = 0.45
DRAW_FRAME_BORDER = True

OUT_DIR = Path("./or_reprint_overlay_outputs")
OUT_DIR.mkdir(exist_ok=True)

In [ ]:
if not Path(FOLDER).exists():
    Path(EXTRACT_TO).mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_FILE, 'r') as zf:
        zf.extractall(EXTRACT_TO)
    print("Extracted to:", EXTRACT_TO)
else:
    print("Folder already exists:", FOLDER)

In [ ]:
def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    import sys
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

notebook_dir = Path('.').resolve()
filament_array = load_module('filament_array', str(notebook_dir / FILAMENT_ARRAY_PY))
gcode_coordinate_mask = load_module('gcode_coordinate_mask', str(notebook_dir / GCODE_COORD_MASK_PY))
merge_miss_print_or = load_module('merge_miss_print_or', str(notebook_dir / MERGE_OR_PY))

print('Loaded:')
print(' -', notebook_dir / FILAMENT_ARRAY_PY)
print(' -', notebook_dir / GCODE_COORD_MASK_PY)
print(' -', notebook_dir / MERGE_OR_PY)

## Build the OR no-filament mask, expected filament mask, and missed-print mask

In [ ]:
out = merge_miss_print_or.merge_miss_print_or(
    folder=FOLDER,
    gcode_file=GCODE_FILE,
    radius=RADIUS,
    threshold=THRESHOLD,
    px_per_mm=PX_PER_MM,
    margin_px=MARGIN_PX,
    flip_x=FLIP_X,
    flip_y=FLIP_Y,
    swap_xy=SWAP_XY,
    layer_index=LAYER_INDEX,
    gcode_line_width_mm=GCODE_LINE_WIDTH_MM,
    max_frame_z=MAX_FRAME_Z,
    merged_out_path=str(OUT_DIR / 'merged_no_filament_or.png'),
    expected_out_path=str(OUT_DIR / 'expected_filament.png'),
    miss_out_path=str(OUT_DIR / 'miss_print_or.png'),
)

no_filament_or = out['no_filament_mask']
expected_filament = out['expected_filament_mask']
miss_print = out['miss_print_mask']
origin_x_mm, origin_y_mm = out['origin_xy_mm']

print('frames_used:', out['frames_used'])
print('origin_xy_mm:', out['origin_xy_mm'])

In [ ]:
def show_bool_mask(mask, title, figsize=(8, 8)):
    plt.figure(figsize=figsize)
    plt.imshow(mask, cmap='gray')
    plt.title(title)
    plt.axis('off')
    plt.show()

show_bool_mask(no_filament_or, 'Global OR no-filament mask')
show_bool_mask(expected_filament, 'Expected filament mask from G-code')
show_bool_mask(miss_print, 'Missed-print mask = expected filament AND OR no-filament')

In [ ]:
fig = plt.figure(figsize=(18, 6))
for i, (mask, title) in enumerate([
    (no_filament_or, 'OR no-filament mask'),
    (expected_filament, 'Expected filament mask'),
    (miss_print, 'Missed-print mask'),
], start=1):
    ax = fig.add_subplot(1, 3, i)
    ax.imshow(mask, cmap='gray')
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
fig.savefig(OUT_DIR / 'mask_triptych.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved:', OUT_DIR / 'mask_triptych.png')

## Parse the G-code and score each extrusion segment

In [ ]:
def sample_segment_pixels(x0, y0, x1, y1, origin_x_mm, origin_y_mm, px_per_mm, shape):
    h, w = shape
    dist_mm = math.hypot(x1 - x0, y1 - y0)
    n = max(2, int(math.ceil(dist_mm * px_per_mm * 1.5)))
    t = np.linspace(0.0, 1.0, n)
    xs = x0 + (x1 - x0) * t
    ys = y0 + (y1 - y0) * t
    cols = np.round((xs - origin_x_mm) * px_per_mm).astype(int)
    rows = np.round((ys - origin_y_mm) * px_per_mm).astype(int)
    good = (rows >= 0) & (rows < h) & (cols >= 0) & (cols < w)
    return rows[good], cols[good]

segments, unsupported = gcode_coordinate_mask.parse_gcode_extrusion_segments(GCODE_FILE)
selected, target_z, layer_zs = gcode_coordinate_mask.pick_layer_segments_by_index(
    segments,
    layer_index=LAYER_INDEX,
    include_previous_layers=False,
    z_tol=0.08,
)
selected = gcode_coordinate_mask.transform_segments_xy(
    selected,
    flip_x=FLIP_X,
    flip_y=FLIP_Y,
    swap_xy=SWAP_XY,
)

segment_scores = []
for idx, seg in enumerate(selected):
    x0, y0, _, x1, y1, _, _, line_no = seg
    rr, cc = sample_segment_pixels(x0, y0, x1, y1, origin_x_mm, origin_y_mm, PX_PER_MM, miss_print.shape)
    vals = miss_print[rr, cc].astype(np.float32) if len(rr) else np.array([0.0], dtype=np.float32)
    severity_mean = float(vals.mean())
    severity_sum = float(vals.sum())
    segment_scores.append({
        'idx': idx,
        'line_no': int(line_no),
        'x0': float(x0), 'y0': float(y0),
        'x1': float(x1), 'y1': float(y1),
        'severity_mean': severity_mean,
        'severity_sum': severity_sum,
        'sample_count': int(len(vals)),
    })

severity_key = 'severity_mean' if SEVERITY_MODE == 'mean' else 'severity_sum'
vals = np.array([s[severity_key] for s in segment_scores], dtype=float)
print('Selected extrusion segments:', len(segment_scores))
print('Severity mode:', severity_key)
print('min  :', vals.min())
print('q25  :', np.quantile(vals, 0.25))
print('q50  :', np.quantile(vals, 0.50))
print('q75  :', np.quantile(vals, 0.75))
print('q90  :', np.quantile(vals, 0.90))
print('max  :', vals.max())

## Render the thresholded faulty segments

In [ ]:
def render_segment_map(expected_filament, segment_scores, severity_key, severity_threshold,
                       origin_x_mm, origin_y_mm, px_per_mm, line_thickness=2):
    img = np.zeros((*expected_filament.shape, 3), dtype=np.uint8)
    img[expected_filament] = (210, 210, 210)

    count = 0
    for s in segment_scores:
        if s[severity_key] < severity_threshold:
            continue
        c0 = int(round((s['x0'] - origin_x_mm) * px_per_mm))
        r0 = int(round((s['y0'] - origin_y_mm) * px_per_mm))
        c1 = int(round((s['x1'] - origin_x_mm) * px_per_mm))
        r1 = int(round((s['y1'] - origin_y_mm) * px_per_mm))
        cv2.line(img, (c0, r0), (c1, r1), (30, 30, 255), line_thickness, cv2.LINE_AA)
        count += 1
    return img, count

segment_img, count = render_segment_map(
    expected_filament,
    segment_scores,
    severity_key=severity_key,
    severity_threshold=SEGMENT_SEVERITY_THRESHOLD,
    origin_x_mm=origin_x_mm,
    origin_y_mm=origin_y_mm,
    px_per_mm=PX_PER_MM,
    line_thickness=SEGMENT_DRAW_THICKNESS,
)

plt.figure(figsize=(10, 10))
plt.imshow(cv2.cvtColor(segment_img, cv2.COLOR_BGR2RGB))
plt.title(f'Reprint segments from OR mask
{severity_key} >= {SEGMENT_SEVERITY_THRESHOLD} | segments={count}')
plt.axis('off')
plt.show()

cv2.imwrite(str(OUT_DIR / 'reprint_segments_from_or_mask.png'), segment_img)
print('Saved:', OUT_DIR / 'reprint_segments_from_or_mask.png')

## Build a global binary mask of the faulty segments

This mask is what will be projected back onto the source images.

In [ ]:
def build_faulty_segment_mask(shape, segment_scores, severity_key, severity_threshold,
                              origin_x_mm, origin_y_mm, px_per_mm, line_thickness=2, pad_px=0):
    mask = np.zeros(shape, dtype=np.uint8)
    thickness = max(1, int(line_thickness + 2 * pad_px))
    for s in segment_scores:
        if s[severity_key] < severity_threshold:
            continue
        c0 = int(round((s['x0'] - origin_x_mm) * px_per_mm))
        r0 = int(round((s['y0'] - origin_y_mm) * px_per_mm))
        c1 = int(round((s['x1'] - origin_x_mm) * px_per_mm))
        r1 = int(round((s['y1'] - origin_y_mm) * px_per_mm))
        cv2.line(mask, (c0, r0), (c1, r1), 255, thickness, cv2.LINE_AA)
    return mask > 0

faulty_segment_mask = build_faulty_segment_mask(
    expected_filament.shape,
    segment_scores,
    severity_key=severity_key,
    severity_threshold=SEGMENT_SEVERITY_THRESHOLD,
    origin_x_mm=origin_x_mm,
    origin_y_mm=origin_y_mm,
    px_per_mm=PX_PER_MM,
    line_thickness=SEGMENT_DRAW_THICKNESS,
    pad_px=FAULTY_SEGMENT_PADDING_PX,
)

plt.figure(figsize=(8,8))
plt.imshow(faulty_segment_mask, cmap='gray')
plt.title('Global faulty-segment mask used for image selection')
plt.axis('off')
plt.show()

cv2.imwrite(str(OUT_DIR / 'faulty_segment_search_mask.png'), faulty_segment_mask.astype(np.uint8)*255)
print('Saved:', OUT_DIR / 'faulty_segment_search_mask.png')

## Helper functions for accurate frame projection

The key inverse operation is `crop_global_mask_to_frame(...)`:

- it converts the frame pose to the same global pixel center used by the merge,
- it extracts a crop from the global faulty-segment mask with the same size as the original image,
- that crop is therefore aligned with the original image pixels.

In [ ]:
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

def list_scan_images(folder):
    names = [f for f in os.listdir(folder) if os.path.splitext(f)[1].lower() in IMG_EXTS]
    return sorted(names, key=merge_miss_print_or.number)


def frame_center_global_pixels(pose, origin_x_mm, origin_y_mm, px_per_mm,
                               flip_x=False, flip_y=True, swap_xy=False):
    x, y = gcode_coordinate_mask.transform_xy(
        pose['x'], pose['y'],
        flip_x=flip_x, flip_y=flip_y, swap_xy=swap_xy,
    )
    cx = int(round((x - origin_x_mm) * px_per_mm))
    cy = int(round((y - origin_y_mm) * px_per_mm))
    return cx, cy


def crop_global_mask_to_frame(global_mask, cx, cy, image_shape):
    H, W = image_shape[:2]
    x0 = cx - W // 2
    y0 = cy - H // 2
    x1 = x0 + W
    y1 = y0 + H

    out = np.zeros((H, W), dtype=global_mask.dtype)

    gx0 = max(0, x0)
    gy0 = max(0, y0)
    gx1 = min(global_mask.shape[1], x1)
    gy1 = min(global_mask.shape[0], y1)

    if gx0 >= gx1 or gy0 >= gy1:
        return out

    ox0 = gx0 - x0
    oy0 = gy0 - y0
    out[oy0:oy0 + (gy1 - gy0), ox0:ox0 + (gx1 - gx0)] = global_mask[gy0:gy1, gx0:gx1]
    return out


def draw_mask_overlay(image_bgr, mask_bool, color_bgr=(0,0,255), alpha=0.45):
    out = image_bgr.copy()
    overlay = image_bgr.copy()
    overlay[mask_bool] = color_bgr
    out = cv2.addWeighted(overlay, alpha, out, 1 - alpha, 0)
    # restore untouched pixels exactly
    out[~mask_bool] = image_bgr[~mask_bool]
    return out

## Select and copy source images that intersect the faulty segments

This section:

1. projects the global faulty-segment mask back to each source image,
2. intersects that with the frame's evaluated region,
3. measures overlap,
4. copies the matching images into a new folder,
5. saves an overlay image with the faulty segments drawn on top.

In [ ]:
FAULTY_IMAGE_OUTPUT_DIR = OUT_DIR / (
    f'faulty_source_images_overlay_{SEVERITY_MODE}_{SEGMENT_SEVERITY_THRESHOLD:g}'
)
FAULTY_IMAGE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

scan_images = list_scan_images(FOLDER)
rows = []
selected_count = 0

for img_i, name in enumerate(scan_images):
    pose = merge_miss_print_or.parse_pose(name)
    if pose is None:
        continue
    if MAX_FRAME_Z is not None and pose['z'] > MAX_FRAME_Z:
        continue

    image_path = os.path.join(FOLDER, name)
    image_bgr = cv2.imread(image_path, cv2.IMREAD_COLOR)
    if image_bgr is None:
        continue

    # Use the same geometry as the extractor merge.
    no_mask, evaluated_mask = merge_miss_print_or.read_masks(FOLDER + os.sep, img_i, RADIUS, THRESHOLD)

    cx, cy = frame_center_global_pixels(
        pose,
        origin_x_mm=origin_x_mm,
        origin_y_mm=origin_y_mm,
        px_per_mm=PX_PER_MM,
        flip_x=FLIP_X,
        flip_y=FLIP_Y,
        swap_xy=SWAP_XY,
    )

    frame_faulty_mask = crop_global_mask_to_frame(faulty_segment_mask, cx, cy, image_bgr.shape)

    # Restrict the overlap test to the actual evaluated strip from the extractor.
    visible_faulty_mask = frame_faulty_mask & evaluated_mask

    overlap_pixels = int(visible_faulty_mask.sum())
    evaluated_pixels = int(evaluated_mask.sum())
    overlap_fraction = float(overlap_pixels / max(evaluated_pixels, 1))

    keep = (
        overlap_pixels >= MIN_FAULTY_OVERLAP_PIXELS
        or overlap_fraction >= MIN_FAULTY_OVERLAP_FRACTION
    )

    rows.append({
        'frame_name': name,
        'x': pose['x'], 'y': pose['y'], 'z': pose['z'], 't': pose['t'],
        'global_cx': cx, 'global_cy': cy,
        'overlap_pixels': overlap_pixels,
        'evaluated_pixels': evaluated_pixels,
        'overlap_fraction': overlap_fraction,
        'selected': keep,
    })

rows = sorted(rows, key=lambda d: (-d['overlap_pixels'], -d['overlap_fraction'], d['frame_name']))

if MAX_IMAGES_TO_COPY is not None:
    rows = rows[:MAX_IMAGES_TO_COPY]

In [ ]:
import pandas as pd

manifest = pd.DataFrame(rows)
manifest_path = FAULTY_IMAGE_OUTPUT_DIR / 'faulty_image_manifest.csv'
manifest.to_csv(manifest_path, index=False)
print('Saved:', manifest_path)
manifest.head(20)

In [ ]:
selected_rows = [r for r in rows if r['selected']]
print('Selected frames:', len(selected_rows))

preview_paths = []
for r in selected_rows[:MAX_IMAGES_TO_COPY if MAX_IMAGES_TO_COPY is not None else len(selected_rows)]:
    image_path = os.path.join(FOLDER, r['frame_name'])
    image_bgr = cv2.imread(image_path, cv2.IMREAD_COLOR)
    if image_bgr is None:
        continue

    pose = merge_miss_print_or.parse_pose(r['frame_name'])
    img_i = scan_images.index(r['frame_name'])
    _, evaluated_mask = merge_miss_print_or.read_masks(FOLDER + os.sep, img_i, RADIUS, THRESHOLD)
    frame_faulty_mask = crop_global_mask_to_frame(
        faulty_segment_mask,
        r['global_cx'], r['global_cy'],
        image_bgr.shape,
    )
    visible_faulty_mask = frame_faulty_mask & evaluated_mask

    overlay_bgr = draw_mask_overlay(
        image_bgr,
        visible_faulty_mask,
        color_bgr=OVERLAY_COLOR_BGR,
        alpha=OVERLAY_ALPHA,
    )

    if DRAW_FRAME_BORDER and visible_faulty_mask.any():
        ys, xs = np.where(visible_faulty_mask)
        x0, x1 = xs.min(), xs.max()
        y0, y1 = ys.min(), ys.max()
        cv2.rectangle(overlay_bgr, (x0, y0), (x1, y1), (0, 255, 255), 1)

    stem = Path(r['frame_name']).stem
    out_name = f"{stem}_faulty_overlay.png"
    out_path = FAULTY_IMAGE_OUTPUT_DIR / out_name
    cv2.imwrite(str(out_path), overlay_bgr)
    preview_paths.append(str(out_path))

print('Overlay images saved in:', FAULTY_IMAGE_OUTPUT_DIR)
print('Saved count:', len(preview_paths))

## Why this overlay is accurate

For each source image:

1. the frame pose from the filename gives the image center in printer coordinates,
2. that center is mapped to global pixels using the same `origin_xy_mm` and `PX_PER_MM` as the merge,
3. a crop of the global faulty-segment mask is taken with exactly the same width and height as the original source image,
4. that crop is aligned pixel-for-pixel with the original image,
5. the overlay is finally restricted to the frame's **evaluated strip** so only the portion that was actually analyzed is highlighted.

This is more accurate than re-estimating scale from the displayed images, because it directly reuses the merge geometry.

## Preview some saved overlay images

In [ ]:
def show_image_grid(image_paths, ncols=3, max_images=9, title='Overlay preview'):
    image_paths = image_paths[:max_images]
    if not image_paths:
        print('No images to preview.')
        return
    n = len(image_paths)
    nrows = int(np.ceil(n / ncols))
    fig = plt.figure(figsize=(5 * ncols, 4 * nrows))
    for i, p in enumerate(image_paths, start=1):
        img = cv2.imread(p, cv2.IMREAD_COLOR)
        ax = fig.add_subplot(nrows, ncols, i)
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(Path(p).name, fontsize=8)
        ax.axis('off')
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

show_image_grid(preview_paths, ncols=3, max_images=9, title='Copied source images with faulty-segment overlays')

## Compare a threshold sweep

This helps tune which segments are considered faulty before image extraction.

In [ ]:
thresholds = [0.05, 0.10, 0.15, 0.20] if SEVERITY_MODE == 'mean' else [5, 10, 20, 40]
fig = plt.figure(figsize=(5 * len(thresholds), 5))

for i, thr in enumerate(thresholds, start=1):
    img, count = render_segment_map(
        expected_filament,
        segment_scores,
        severity_key=severity_key,
        severity_threshold=thr,
        origin_x_mm=origin_x_mm,
        origin_y_mm=origin_y_mm,
        px_per_mm=PX_PER_MM,
        line_thickness=SEGMENT_DRAW_THICKNESS,
    )
    ax = fig.add_subplot(1, len(thresholds), i)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f'{severity_key} >= {thr}
segments={count}')
    ax.axis('off')

plt.tight_layout()
fig.savefig(OUT_DIR / 'reprint_segment_threshold_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved:', OUT_DIR / 'reprint_segment_threshold_comparison.png')